In [59]:
import os
import pandas as pd
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
import json
import tiktoken

REPO_DIR = os.path.join("/Users/haya1/Documents/LanguageModel_Labels/congressional_bills/")
# REPO_DIR = "."
os.chdir(REPO_DIR)

load_dotenv(os.path.join(REPO_DIR, ".env"), override=True)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')

In [58]:
def create_embed_requests(df_descriptions, out_dir, embedding_model="text-embedding-3-small"):
    batches = []
    requests = []
    
    I = len(df_descriptions)
    for col_name in df_descriptions.columns:
        description_col = df_descriptions[col_name]
        for i in range(I):
            description = description_col.loc[i]
        
            request = {
                "custom_id": str(i),
                "method": "POST",
                "url": "/v1/embeddings",
                "body": {
                    "input": description,
                    "model": embedding_model
                }
            }
            requests.append(request)

            if (i==(I-1)):
                col_path = os.path.join(out_dir, f"requests_{col_name}.jsonl")

                with open(col_path, "w") as f:
                    for request in requests:
                        f.write(json.dumps(request) + "\n")
                    print(f"Saved {os.path.basename(col_path)}, n = {len(requests)}, at {os.path.dirname(col_path)}")
                requests = []

                batch = {
                    'file': col_path,
                    'col_name': col_name
                }
                batches.append(batch)

    return(pd.json_normalize(batches))

In [60]:
def query_embeddings(batches):
    client = OpenAI(api_key=OPENAI_API_KEY)

    batches_id = []
    for _, batch in batches.iterrows():
        batch_input_file = client.files.create(
            file = open(batch['file'], "rb"),
            purpose = "batch"
        )

        new_batch = client.batches.create(
            input_file_id = batch_input_file.id,
            endpoint = "/v1/embeddings",
            completion_window = "24h",
            metadata = {"description": f"{os.path.basename(batch['file'])}"}
        )

        batches_id.append(new_batch.id)

    batches['id'] = batches_id
    return(batches)

In [61]:
# Check status of all batches
def check_batches_status(batches):
    client = OpenAI(api_key=OPENAI_API_KEY)
    for _, batch in batches.iterrows():
        file = batch['file']
        id = batch['id']
        batch_status = client.batches.retrieve(id)
        print(f"{os.path.basename(file):>52s}: {batch_status.status}")


In [62]:
def download_batched_responses(batches, out_dir):
    client = OpenAI(api_key=OPENAI_API_KEY)
    for _, batch in batches.iterrows():
        batch_id = batch['id']
        col_name = batch['col_name']
        responses_batched_path = os.path.join(out_dir, f"responses_{col_name}.jsonl")

        batch_status = client.batches.retrieve(batch_id)
        if batch_status.status != "completed":
            print(f"Skipping incomplete file = {os.path.basename(responses_batched_path)}, Batch ID = {batch_id}")
            continue
        
        output_file_id = batch_status.output_file_id
        responses_batched = client.files.content(output_file_id)
        responses_batched.write_to_file(responses_batched_path)
        print(f"Saved {os.path.basename(responses_batched_path)} at {os.path.dirname(responses_batched_path)}")

In [63]:
def count_tokens(text, model="text-embedding-3-small"):
    encoding = tiktoken.encoding_for_model(model)
    n_tokens = len(encoding.encode(text)) 
    return(n_tokens)

def estimate_cost(descriptions, model="text-embedding-3-small", batched=True):
    n_tokens = np.zeros_like(descriptions)
    for i in range(len(descriptions)):
        n_tokens[i] = count_tokens(descriptions[i], model)
    
    # Cost without using Batch API
    embed_token_cost = {
        'text-embedding-3-small': 0.020/1e6,
        'text-embedding-3-large': 0.130/1e6,
        'ada v2': 0.100/1e6
    }

    total_cost  = (n_tokens * embed_token_cost[model]).sum()
    if batched:
        total_cost = total_cost/2
    return total_cost

In [64]:
data_dir = os.path.join(REPO_DIR, "Data/Prediction_run1")
temp_dir = os.path.join(REPO_DIR, "Temp/Prediction_run1")

bills_llm_completion = pd.read_csv(os.path.join(data_dir, "bills_llm_completion.csv"))

In [65]:
cost_description = estimate_cost(bills_llm_completion["DescriptionClean"], batched=True)
print(f"Estimated cost to embed Description using Batch API is ${cost_description:.2f}")

cost_description_llm = estimate_cost(bills_llm_completion["DescriptionLLMClean"], batched=True)
print(f"Estimated cost to embed DescriptionLLM using Batch API is ${cost_description_llm:.2f}")

Estimated cost to embed Description using Batch API is $0.00
Estimated cost to embed DescriptionLLM using Batch API is $0.01


In [66]:
requests_dir = os.path.join(temp_dir, "Embeddings/Requests")
os.makedirs(requests_dir, exist_ok=True)

batches = create_embed_requests(bills_llm_completion[["DescriptionClean", "DescriptionLLMClean"]], requests_dir, embedding_model="text-embedding-3-small")
batches_path = os.path.join(temp_dir, "Embeddings/batches.csv")
batches.to_csv(batches_path, index=False)
print(f"Created batched prompts with batch details stored at {os.path.basename(batches_path)}, n = {len(batches)}, at {os.path.dirname(batches_path)}")

Saved requests_DescriptionClean.jsonl, n = 39999, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/Embeddings/Requests
Saved requests_DescriptionLLMClean.jsonl, n = 39999, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/Embeddings/Requests
Created batched prompts with batch details stored at batches.csv, n = 2, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/Embeddings


In [67]:
batches = query_embeddings(batches)
batches.to_csv(batches_path, index=False)
print(f"Added batch_id to {os.path.basename(batches_path)}, n = {len(batches)}, at {os.path.dirname(batches_path)}")

Added batch_id to batches.csv, n = 2, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/Embeddings


In [82]:
batches = pd.read_csv(os.path.join(temp_dir, "Embeddings/batches.csv"))
check_batches_status(batches)

                     requests_DescriptionClean.jsonl: completed
                  requests_DescriptionLLMClean.jsonl: completed


In [83]:
batches = pd.read_csv(os.path.join(temp_dir, "Embeddings/batches.csv"))
responses_dir = os.path.join(temp_dir, "Embeddings/Responses")
os.makedirs(responses_dir, exist_ok=True)
download_batched_responses(batches, responses_dir)

Saved responses_DescriptionClean.jsonl at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/Embeddings/Responses
Saved responses_DescriptionLLMClean.jsonl at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/Embeddings/Responses
